In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

e:\Agents\agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Ahmed i am a cheif ai engineer")])

AIMessage(content="Hello Ahmed, nice to meet you. As a chief AI engineer, I'm sure you have a deep understanding of the latest advancements and applications of artificial intelligence. What brings you here today? Are you working on a specific project or looking for insights on the latest AI trends?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 48, 'total_tokens': 104, 'completion_time': 0.105831458, 'completion_tokens_details': None, 'prompt_time': 0.002576559, 'prompt_tokens_details': None, 'queue_time': 0.04693455, 'total_time': 0.108408017}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019faa3b-728a-7333-91ba-59da25e392d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 56, 'total_tokens': 104})

### basic single session wrapper-> [] memeory!!!

In [4]:
# from langchain_core.messages import AIMessage
# #list of messages
# model.invoke(
#     [
#         HumanMessage(content="Hi, My name is Ahmed i am a cheif ai engineer"),
#         AIMessage(content="Nice to meet you, Ahmed. As a Chief AI Engineer, you must be working on some exciting projects. Can you tell me a bit about what you're currently working on or any interesting challenges you're facing in the field of AI?.\n"),
#         HumanMessage(content="Hey What's my nmae and what do i do?")
#     ]
# )

### chat_message_history , basicchatmessagehistory, runnablewithmessagehistory

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
#func to check if its in the history then add if not there!
def get_session_history(session_id:str)->BaseChatMessageHistory: #for storing chat message history
    if session_id not in store:
        store[session_id]=ChatMessageHistory() #store message in memory also rertrive
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

# model — your LLM/chat model (e.g. ChatOpenAI(), ChatGroq(), etc.) — the actual "brain" that generates responses.
# get_session_history — the function you wrote, passed in by reference (not called). RunnableWithMessageHistory will call it internally whenever it needs a session's history.
# with_message_history — a new runnable that behaves just like model, but now automatically wraps every call with history: fetch past messages → send to model → save the new exchange back.

C:\Users\Moham\AppData\Local\Temp\ipykernel_196\3206833129.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
e:\Agents\agent\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
# ChatMessageHistory() — this is a simple object that holds a list of messages
#  (HumanMessage, AIMessage, etc.) 
# for one conversation, and exposes methods like .add_user_message(), 
# .add_ai_message(), and .messages to retrieve them all.

In [7]:
#hard coding tracking id 
config={"configurable":{"session_id":"chat1"}}

In [8]:
resp=with_message_history.invoke(
    [HumanMessage(content="Hi, my name ahmed and i am a chief ai engineer")],
    config=config
)

resp.content

"Nice to meet you, Ahmed. As a Chief AI Engineer, that's a fascinating role. I'm curious to know more about your work and what you're currently focused on in the AI space. What areas of AI interest you the most, such as natural language processing, computer vision, or perhaps reinforcement learning?"

In [9]:
#change session_id(config)
config3={"configurable":{"session_id":"chat3"}}
resp3=with_message_history.invoke(
    [HumanMessage(content="what is my name adn what i do?")],
    config=config3
)

resp3.content

"Unfortunately, I don't have any information about you. I'm a conversational AI, and our conversation just started. I don't have any prior knowledge or context about you. If you'd like to share your name and what you do, I'd be happy to chat with you!"

In [10]:
resp4=with_message_history.invoke(
    [HumanMessage(content="Whats my name and what i do?")],
    config=config
)

resp4.content #it remebers from config history!

"Your name is Ahmed, and you're a Chief AI Engineer."

### caht prompt template & message history

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Use the conversation history exactly as given. "
            "If the user states their name, remember it exactly. "
            "Do not invent or change the name. If the name is not known, say you do not know."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_arg="messages",
)


e:\Agents\agent\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [12]:
chain.invoke({"messages": [HumanMessage(content="My name is Ahmed, just remember it.")]})

AIMessage(content="I've noted your name as Ahmed. What can I assist you with today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 90, 'total_tokens': 107, 'completion_time': 0.039797832, 'completion_tokens_details': None, 'prompt_time': 0.007265812, 'prompt_tokens_details': None, 'queue_time': 0.047417678, 'total_time': 0.047063644}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019faa3b-78f4-7832-8f93-fb2ff04af849-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 17, 'total_tokens': 107})

In [13]:
# wrap it with with_message_history using the prompt chain
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_arg="messages",
)


e:\Agents\agent\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
config7 = {"configurable": {"session_id": "chat7"}}
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="My name is Ahmed, please remember it.")]},
    config=config7,
)

print(response.content)


I've remembered your name as Ahmed. How can I assist you today?


In [15]:
### adding a little complexicity
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all question to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [16]:
response2=chain.invoke({"messages":[HumanMessage(content="My name is MTA")],"language":"english"})
response2.content

'Nice to meet you, MTA. Is there something I can help you with today?'

In [17]:
#wrap it around message history
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

e:\Agents\agent\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
config8={"configurable":{"session_id":"chat8"}}
response2=with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name")], "language":"english"},
    config=config8
)
response2.content

"I don't have any information about your name. We just started our conversation, and I'm a new conversational partner for you. If you'd like, you can share your name with me, and I'll be happy to refer to it in our conversation."

### Manage the convo history, coz u dont want to go beyond the context window and also add limit the size of the message you are passing!

In [22]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage, trim_messages
trimmer_messages=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human" 
)

#set of messages
messages=[
    SystemMessage(content="You're a good assistant"),
    HumanMessage(content="hi, my name is mohammed!"),
    AIMessage(content="hi!"),
    HumanMessage(content="i like ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 5*5"),
    AIMessage(content="its 25"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun ?"),
    AIMessage(content="yes!")
]

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain2=RunnablePassthrough.assign(
    messages=itemgetter("messages") | trimmer_messages
    ) | prompt | model

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="do u remeber our convo anything?")],
    "language": "english"
    }
)
response.content

"This is the start of our conversation, so there's nothing to remember yet! We just exchanged a few pleasantries. I'm here to help with any questions or topics you'd like to discuss. What's on your mind?"

In [29]:
# assign() is a method on Runnable objects in LangChain. 
# It adds or computes new keys in the input dictionary while keeping the existing ones

In [30]:
#wrap this message to history
with_message_history=RunnableWithMessageHistory(
    chain2,
    get_session_history,
    input_messages_key="messages"
)

config10={'configurable':{"session_id":"chat10"}}
response3=with_message_history.invoke(
    {"messages": [HumanMessage(content="what was my last question")], "language":"english"},
    config=config10
)
response3.content


e:\Agents\agent\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


"This is the beginning of our conversation, so you haven't asked a question yet. What would you like to talk about? I'm here to help."